# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Mounted at /content/drive
GitHub Token: ··········
Cloning into 'RecSys-Challenge-2025'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (376/376), done.
remote: Compressing objects: 100% (283/283), done.
remote: Total 376 (delta 169), reused 284 (delta 84), pack-reused 0 (from 0)
Receiving objects: 100% (376/376), 7.45 MiB | 20.46 MiB/s, done.
Resolving deltas: 100% (169/169), done.


In [2]:
# Compile Cython files
os.chdir(LOCAL_REPO_PATH)
!python Challenge/compile_cython.py
os.chdir(WORKING_DIR)

Compiling /content/RecSys-Challenge-2025/Recommenders/FeatureWeighting/Cython/CFW_D_Similarity_Cython_SGD.pyx because it changed.
Compiling /content/RecSys-Challenge-2025/Recommenders/FeatureWeighting/Cython/CFW_DVV_Similarity_Cython_SGD.pyx because it changed.
Compiling /content/RecSys-Challenge-2025/Recommenders/FeatureWeighting/Cython/FBSM_Rating_Cython_SGD.pyx because it changed.
Compiling /content/RecSys-Challenge-2025/Recommenders/FeatureWeighting/Cython/HP3_Similarity_Cython_SGD.pyx because it changed.
Compiling /content/RecSys-Challenge-2025/Recommenders/Similarity/Cython/Compute_Similarity_Cython.pyx because it changed.
Compiling /content/RecSys-Challenge-2025/Recommenders/SLIM/Cython/Triangular_Matrix.pyx because it changed.
Compiling /content/RecSys-Challenge-2025/Recommenders/SLIM/Cython/SLIM_BPR_Cython_Epoch.pyx because it changed.
Compiling /content/RecSys-Challenge-2025/Recommenders/SLIM/Cython/Sparse_Matrix_Tree_CSR.pyx because it changed.
Compiling /content/RecSys-Chal

In [3]:
%%capture
if not IS_LOCAL:
    !pip install optuna

import optuna

In [4]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on colab — storage at: /content/drive/MyDrive/RecSys
Running on colab — storage at: /content/drive/MyDrive/RecSys


# **Load data**

In [5]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [6]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0

    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]

        if len(relevant_items)>0:
            num_eval+=1

            recommended_items = recommender.recommend(user_id, cutoff=at)

            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Train a KNN with Jaccard similarity**

In [7]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

In [8]:
def perform_optimization(similarity, n_trials):
    # Define objective function for hyperparameter tuning
    STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_" + similarity

    def objective_function(optuna_trial: optuna.trial.Trial) -> float:
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(
            similarity=similarity,
            topK=optuna_trial.suggest_int("topK", 10, 1500),
            shrink=optuna_trial.suggest_int("shrink", 0, 2000),
            normalize=optuna_trial.suggest_categorical("normalize", [True, False]),
            feature_weighting=optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
        )

        return evaluate_recommender(recommender_instance, at=20)

    # Perform hyperparameter tuning
    save_results, optuna_study = hyperparameter_tuning(
        objective_function,
        study_name=STUDY_NAME,
        n_trials=n_trials
    )

    return save_results, optuna_study

## **Hyperparameter Tuning**

In [9]:
SIMILARITY = "jaccard"

In [10]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 100)

  0%|          | 0/100 [00:00<?, ?it/s]

Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 380.91 column/sec. Elapsed time 18.30 sec
[I 2025-11-08 15:25:30,269] Trial 0 finished with value: 0.13708816468715668 and parameters: {'topK': 797, 'shrink': 320, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 0 with value: 0.13708816468715668.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 318.45 column/sec. Elapsed time 21.88 sec
[I 2025-11-08 15:26:47,846] Trial 1 finished with value: 0.13537317514419556 and parameters: {'topK': 1320, 'shrink': 789, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 0 with value: 0.13708816468715668.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 571.22 column/sec. Elapsed time 12.20 sec
[I 2025-11-08 15:27:30,567] Trial 2 finished with value: 0.14845620095729828 and parameters: {'topK': 164, 'shrink': 847, 'normalize': False, 

In [11]:
optuna.visualization.plot_optimization_history(optuna_study)

In [12]:
optuna.visualization.plot_param_importances(optuna_study)

In [13]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [15]:
STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + "_tuning_" + SIMILARITY

def jaccard_tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = ItemKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 1000, 1200),
        shrink=optuna_trial.suggest_int("shrink", 0, 20),
        normalize=True,
        feature_weighting="BM25"
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    jaccard_tuning_function,
    study_name=STUDY_NAME,
    n_trials=20
)

  0%|          | 0/20 [00:00<?, ?it/s]

Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 315.22 column/sec. Elapsed time 22.11 sec
[I 2025-11-08 17:15:32,816] Trial 0 finished with value: 0.21537092328071594 and parameters: {'topK': 1065, 'shrink': 20}. Best is trial 0 with value: 0.21537092328071594.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 335.51 column/sec. Elapsed time 20.77 sec
[I 2025-11-08 17:16:36,369] Trial 1 finished with value: 0.2149236798286438 and parameters: {'topK': 1161, 'shrink': 4}. Best is trial 0 with value: 0.21537092328071594.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 320.36 column/sec. Elapsed time 21.75 sec
[I 2025-11-08 17:17:39,128] Trial 2 finished with value: 0.2149849385023117 and parameters: {'topK': 1041, 'shrink': 6}. Best is trial 0 with value: 0.21537092328071594.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity colu

# **Best Params**

- Best Value: 0.2154081016778946
- Best Params: {'topK': 1101, 'shrink': 0, 'normalize': True, 'feature_weighting': 'BM25'}